In [1]:
# Cell 1: import library
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt 
from pathlib import Path
import lightgbm as lgb 

import json, time, warnings

In [2]:
# Cell 2: loda dataset & setup
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 60)

bm = json.loads(Path("../data/processed/features_blocks_meta.json").read_text())
H, LAST_DAY, TARGET = bm["H"], bm["LAST_DAY"], bm["TARGET"]
NUM_B, CAT_B = bm["NUM"], bm["CAT"]
FEATS = NUM_B + CAT_B
BLOCKS = {int(k): v for k, v in bm["blocks"].items()}
TEST_START = LAST_DAY - H + 1
CUTOFF = TEST_START - 1

base = pd.read_parquet("../data/processed/features.parquet", columns=["d_num", "date", "item_id", "cat_id", "sales"])
base["sales"] = base["sales"].astype("float32")
Y = base.pivot(index="item_id", columns="d_num", values="sales").reindex(columns=range(1, LAST_DAY + 1))
Y_np = Y.to_numpy()
pos_map = pd.Series(np.arange(len(Y)), index=Y.index.astype(str))
cat_arr = (base.drop_duplicates("item_id").set_index("item_id")["cat_id"]
           .reindex(Y.index).astype(str).to_numpy())
date_of = base.drop_duplicates("d_num").set_index("d_num")["date"]


print("Y shape:", Y.shape, "| total units:", int(np.nansum(Y_np))) 
print("cutoff d_%d = %s | test window d_%d to d_%d = %s to %s" % (
    CUTOFF, date_of[CUTOFF].date(), TEST_START, LAST_DAY,
    date_of[TEST_START].date(), date_of[LAST_DAY].date()))

print("test cells that exist (items launched):", int((~np.isnan(Y_np[:, CUTOFF:])).sum()))
print("blocks:", BLOCKS, "| features:", len(NUM_B), "numeric +", len(CAT_B), "categorical")   
for s in BLOCKS.values():
    f = Path(f"../data/processed/features_s{s}.parquet")

    assert f.exists(), f"missing {f}"
print("all four block tables found in")

Y shape: (3049, 1941) | total units: 7832248
cutoff d_1913 = 2016-04-24 | test window d_1914 to d_1941 = 2016-04-25 to 2016-05-22
test cells that exist (items launched): 85372
blocks: {1: 7, 2: 14, 3: 21, 4: 28} | features: 26 numeric + 7 categorical
all four block tables found in


In [4]:
# Cell 3: Train the final models and predict the test window
PARAMS = dict(learning_rate=0.1, num_leaves=63, min_data_in_leaf=100,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1,
              seed=42, verbose=-1, force_row_wise=True)
MODELS = {"poisson": ({**PARAMS, "objective": "poisson"}, 300),
          "q90": ({**PARAMS, "objective": "quantile", "alpha": 0.9}, 200),
          "q50": ({**PARAMS, "objective": "quantile", "alpha": 0.5}, 200),
          "q10": ({**PARAMS, "objective": "quantile", "alpha": 0.1}, 200)}
n_items = len(Y)
CKPT =  Path("../data/processed/test_preds_checkpoint.npz")

if CKPT.exists():
    z = np.load(CKPT)
    P = {m: z[m] for m in MODELS}
    done = set(int(x) for x in z["done"])
    print("resuming; blocks already done:", sorted(done))
else:
    P = {m: np.full((n_items, H), np.nan, dtype="float32") for m in MODELS}
    done = set()

cols = ["d_num", "sales"] + FEATS
t_all = time.time()
for b, s in BLOCKS.items():
    if b in done:
        continue
    t0 = time.time()
    d0, d1 = TEST_START + 7 * (b - 1), TEST_START + 7 * b - 1
    assert d1 - s <= CUTOFF, "features would reach beyond the cutoff"
    tbl = pd.read_parquet(Path(f"../data/processed/features_s{s}.parquet"), columns=cols)
    train = tbl[tbl["d_num"] <= CUTOFF]
    assert train["d_num"].max() == CUTOFF, "training rows must end at the cutoff"
    val = tbl[(tbl["d_num"] >= d0) & (tbl["d_num"] <= d1)]
    rows_i = pos_map[val["item_id"].astype(str)].to_numpy()
    cols_j = val["d_num"].to_numpy() - TEST_START
    for m, (params, rounds) in MODELS.items():
        dtrain = lgb.Dataset(train[FEATS], label=train["sales"],
                             categorical_feature=CAT_B, free_raw_data=True)
        booster = lgb.train(params, dtrain, num_boost_round=rounds)
        P[m][rows_i, cols_j] = booster.predict(val[FEATS])
        del dtrain, booster
    done.add(b)
    np.savez_compressed(CKPT, done=np.array(sorted(done)), **P)
    print(f"block {b} (shift {s}): train rows {len(train)}, test rows {len(val)} | {time.time() - t0:.0f}s")
    del tbl, train, val
print(f"total this run: {(time.time() - t_all) / 60:.1f} min")

assert done == set(BLOCKS), "not all blocks are done"

# sort the quantiles per cell, then save the final predictions
S = np.sort(np.stack([P["q10"], P["q50"], P["q90"]]), axis=0)
ok = ~np.isnan(P["q10"])
print("crossing before sorting: P10>P50", round(float(np.mean(P["q10"][ok] > P["q50"][ok])), 4),
      "| P50>P90", round(float(np.mean(P["q50"][ok] > P["q90"][ok])), 4))
FINAL = {"poisson": P["poisson"], "q10": S[0], "q50": S[1], "q90": S[2]}
np.savez_compressed(Path("../data/processed/test_pred_final.npz"), **FINAL)


# checks that do not look at sales values
act_exists = ~np.isnan(Y_np[:, CUTOFF:])
for m, arr in FINAL.items():
    print(f"{m:8s} | actual but no prediction: {int((act_exists & np.isnan(arr)).sum())}"
          f" | prediction but no actual: {int((~act_exists & ~np.isnan(arr)).sum())}"
          f" | min prediction: {np.nanmin(arr):.4f}")
print("saved:", (Path("../data/processed/test_pred_final.npz")).resolve())

block 1 (shift 7): train rows 4702895, test rows 21343 | 148s
block 2 (shift 14): train rows 4702895, test rows 21343 | 163s
block 3 (shift 21): train rows 4702895, test rows 21343 | 161s
block 4 (shift 28): train rows 4702895, test rows 21343 | 168s
total this run: 10.7 min
crossing before sorting: P10>P50 0.0216 | P50>P90 0.0031
poisson  | actual but no prediction: 0 | prediction but no actual: 0 | min prediction: 0.0077
q10      | actual but no prediction: 0 | prediction but no actual: 0 | min prediction: -0.3319
q50      | actual but no prediction: 0 | prediction but no actual: 0 | min prediction: -0.0041
q90      | actual but no prediction: 0 | prediction but no actual: 0 | min prediction: 0.0000
saved: C:\Users\ASUS\Desktop\AI_Projects\Retail-demand-forecasting\data\processed\test_pred_final.npz


In [5]:
# Cell 4: Score the test window, once
def _valid(y, p):
    m = ~np.isnan(y) & ~np.isnan(p)
    return y[m], p[m]
def mae(y, p):
    y, p = _valid(y, p); return float(np.abs(y - p).mean())
def rmse(y, p):
    y, p = _valid(y, p); return float(np.sqrt(((y - p) ** 2).mean()))
def wape(y, p):
    y, p = _valid(y, p); return float(np.abs(y - p).sum() / y.sum())
def mase_scale(Y_train, m=7):
    d = np.abs(Y_train[:, m:] - Y_train[:, :-m])
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.nanmean(d, axis=1)
def mase(y, p, scale):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        ratio = np.nanmean(np.abs(y - p), axis=1) / scale
    ok = np.isfinite(ratio) & (scale > 0)
    return float(ratio[ok].mean())
def score(Y_val, P, Y_train):
    return {"MAE": mae(Y_val, P), "RMSE": rmse(Y_val, P), "WAPE": wape(Y_val, P),
            "MASE": mase(Y_val, P, mase_scale(Y_train))}

def zeros(Yt, h=H): return np.zeros((Yt.shape[0], h), dtype="float32")
def naive(Yt, h=H): return np.nan_to_num(np.repeat(Yt[:, -1][:, None], h, axis=1), nan=0.0)
def seasonal_naive(Yt, h=H, m=7):
    return np.nan_to_num(np.tile(Yt[:, -m:], (1, int(np.ceil(h / m))))[:, :h], nan=0.0)
def _rep(v, h): return np.repeat(np.nan_to_num(v, nan=0.0)[:, None], h, axis=1)
def moving_average(Yt, h=H, w=28):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return _rep(np.nanmean(Yt[:, -w:], axis=1), h)
def moving_median(Yt, h=H, w=28):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return _rep(np.nanmedian(Yt[:, -w:], axis=1), h)
def emp_q(Yt, a, h=H, w=28):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return _rep(np.nanquantile(Yt[:, -w:], a, axis=1), h)

# load the saved predictions, clip the quantiles at 0
z = np.load(Path("../data/processed/test_pred_final.npz"))
FINAL = {k: z[k] for k in z.files}
for m in ("q10", "q50", "q90"):
    print(f"{m}: cells below 0: {int((FINAL[m] < 0).sum())} of {FINAL[m].size} | min {FINAL[m].min():.4f}")
Q = {m: np.maximum(FINAL[m], 0) for m in ("q10", "q50", "q90")}

Y_train, Y_test = Y_np[:, :CUTOFF], Y_np[:, CUTOFF:]
assert Y_train.shape[1] == 1913 and Y_test.shape == (len(Y), H)
print("total actual units in the test window:", int(np.nansum(Y_test)))

# point forecasts
P_pt = {"zeros": zeros(Y_train), "naive": naive(Y_train), "seasonal naive (7d)": seasonal_naive(Y_train),
        "moving average (28d)": moving_average(Y_train), "moving median (28d)": moving_median(Y_train),
        "LightGBM Poisson (final)": FINAL["poisson"], "LightGBM P50 (clipped)": Q["q50"]}
test_tbl = pd.DataFrame({n: score(Y_test, P, Y_train) for n, P in P_pt.items()}).T
print("\n=== TEST window (d_1914-d_1941) ===")
display(test_tbl.round(3))

val_ref = pd.DataFrame(
    {"zeros": [1.468, 3.760, 1.000, 0.920], "naive": [1.533, 3.393, 1.045, 1.161],
     "seasonal naive (7d)": [1.288, 2.791, 0.877, 1.022], "moving average (28d)": [1.099, 2.257, 0.748, 0.895],
     "moving median (28d)": [1.024, 2.360, 0.698, 0.771],
     "LightGBM Poisson (final)": [1.084, 2.181, 0.739, 0.899]},
    index=["MAE", "RMSE", "WAPE", "MASE"]).T
print("validation, mean of 3 folds, all items (Notebooks 03 and 06):")
display(val_ref)

# levels: item-day, category-day, store-day
cats = sorted(set(cat_arr))
def levels(P):
    a_c = np.vstack([Y_test[cat_arr == c].sum(axis=0) for c in cats])
    p_c = np.vstack([P[cat_arr == c].sum(axis=0) for c in cats])
    return {"item-day": wape(Y_test, P),
            "category-day": float(np.abs(a_c - p_c).sum() / a_c.sum()),
            "store-day": float(np.abs(Y_test.sum(0) - P.sum(0)).sum() / Y_test.sum())}
lv = pd.DataFrame({n: levels(P_pt[n]) for n in ("moving average (28d)", "seasonal naive (7d)", "LightGBM Poisson (final)")}).T
print("\nWAPE by level, test (validation: LightGBM Poisson+item_id 0.739 / 0.067 / 0.052, "
      "moving average 0.748 / 0.171 / 0.164, seasonal naive 0.877 / 0.119 / 0.102):")
display(lv.round(3))
print("Poisson total predicted / total actual:", round(float(FINAL["poisson"].sum() / Y_test.sum()), 3),
      "(validation 0.997)")

# quantiles
def pinball(y, p, a):
    d = y - p
    return float(np.mean(np.maximum(a * d, (a - 1) * d)))
y = Y_test.ravel().astype("float64")
rows = []
for a, name in ((0.1, "q10"), (0.5, "q50"), (0.9, "q90")):
    for label, Pq in (("LightGBM (clipped)", Q[name]), ("empirical 28d", emp_q(Y_train, a))):
        p = Pq.ravel().astype("float64")
        rows.append({"quantile": a, "model": label, "pinball": pinball(y, p, a),
                     "P(y < pred)": float(np.mean(y < p)), "P(y <= pred)": float(np.mean(y <= p)),
                     "total pred / total actual": float(p.sum() / y.sum())})
print("\nquantiles, test (validation pinball LightGBM / empirical: P10 0.142 / 0.144, "
      "P50 0.496 / 0.512, P90 0.323 / 0.354; P90 coverage 0.899):")
display(pd.DataFrame(rows).round(3))

q10: cells below 0: 3944 of 85372 | min -0.3319
q50: cells below 0: 16 of 85372 | min -0.0041
q90: cells below 0: 0 of 85372 | min 0.0000
total actual units in the test window: 134032

=== TEST window (d_1914-d_1941) ===


,MAE,RMSE,WAPE,MASE
zeros,1.570,3.925,1.000,1.005
naive,1.568,3.055,0.999,1.269
seasonal naive (7d),1.363,2.737,0.868,1.112
moving average (28d),1.174,2.347,0.748,0.976
moving median (28d),1.106,2.452,0.704,0.852
LightGBM Poisson (final),1.128,2.202,0.718,0.961
LightGBM P50 (clipped),1.062,2.304,0.676,0.836


validation, mean of 3 folds, all items (Notebooks 03 and 06):


,MAE,RMSE,WAPE,MASE
zeros,1.468,3.760,1.000,0.920
naive,1.533,3.393,1.045,1.161
seasonal naive (7d),1.288,2.791,0.877,1.022
moving average (28d),1.099,2.257,0.748,0.895
moving median (28d),1.024,2.360,0.698,0.771
LightGBM Poisson (final),1.084,2.181,0.739,0.899



WAPE by level, test (validation: LightGBM Poisson+item_id 0.739 / 0.067 / 0.052, moving average 0.748 / 0.171 / 0.164, seasonal naive 0.877 / 0.119 / 0.102):


,item-day,category-day,store-day
moving average (28d),0.748,0.160,0.151
seasonal naive (7d),0.868,0.087,0.072
LightGBM Poisson (final),0.718,0.068,0.052


Poisson total predicted / total actual: 0.949 (validation 0.997)

quantiles, test (validation pinball LightGBM / empirical: P10 0.142 / 0.144, P50 0.496 / 0.512, P90 0.323 / 0.354; P90 coverage 0.899):


,quantile,model,pinball,P(y < pred),P(y <= pred),total pred / total actual
0,0.1,LightGBM (clipped),0.152,0.011,0.511,0.064
1,0.1,empirical 28d,0.152,0.017,0.523,0.183
2,0.5,LightGBM (clipped),0.531,0.324,0.604,0.689
3,0.5,empirical 28d,0.553,0.213,0.665,0.716
4,0.9,LightGBM (clipped),0.355,0.883,0.885,2.153
5,0.9,empirical 28d,0.392,0.727,0.891,2.006


In [6]:
# Cell 5: How much does the item sample matter? (paired bootstrap)

rng = np.random.default_rng(42)
A = Y_test.astype("float64")
n = A.shape[0]
B = 2000
idx = rng.integers(0, n, size=(B, n))
act_item = A.sum(axis=1)

def wape_item(P):
    return np.abs(A - P.astype("float64")).sum(axis=1)

def pin_item(P, a):
    d = A - P.astype("float64")
    return np.maximum(a * d, (a - 1) * d).sum(axis=1)

pairs = [("WAPE Poisson - moving average", wape_item(FINAL["poisson"]), wape_item(P_pt["moving average (28d)"]), True),
         ("WAPE P50 - moving median", wape_item(Q["q50"]), wape_item(P_pt["moving median (28d)"]), True),
         ("pinball P50 - empirical", pin_item(Q["q50"], 0.5), pin_item(emp_q(Y_train, 0.5), 0.5), False),
         ("pinball P90 - empirical", pin_item(Q["q90"], 0.9), pin_item(emp_q(Y_train, 0.9), 0.9), False)]
rows = []
for name, e1, e2, is_wape in pairs:
    den = act_item[idx].sum(axis=1) if is_wape else idx.shape[1] * H
    d = (e1[idx].sum(axis=1) - e2[idx].sum(axis=1)) / den
    point = (e1.sum() - e2.sum()) / (act_item.sum() if is_wape else n * H)
    rows.append({"difference": name, "point": point, "2.5%": np.percentile(d, 2.5),
                 "97.5%": np.percentile(d, 97.5), "share of samples LightGBM better": float((d < 0).mean())})
display(pd.DataFrame(rows).round(4))

,difference,point,2.5%,97.5%,share of samples LightGBM better
0,WAPE Poisson - moving average,-0.0296,-0.0357,-0.0236,1.0
1,WAPE P50 - moving median,-0.0277,-0.0337,-0.0216,1.0
2,pinball P50 - empirical,-0.0218,-0.0274,-0.0164,1.0
3,pinball P90 - empirical,-0.0375,-0.0432,-0.0320,1.0


In [7]:
# Cell 6: Inventory simulation on the test window

D_all = Y_test.ravel().astype("float64")
def summarize(S, D):
    short, over, sd = np.maximum(D - S, 0), np.maximum(S - D, 0), D > 0
    return {"fill rate": 1 - short.sum() / D.sum(), "stockout units (% of demand)": 100 * short.sum() / D.sum(),
            "short on sales days (%)": 100 * np.mean((D > S)[sd]),
            "overstock units (% of demand)": 100 * over.sum() / D.sum(), "stock / demand": S.sum() / D.sum()}
def stock_k(p, k): return np.floor(k * p + 0.5)
def scale_to(p, target):
    lo, hi = 0.0, 1000.0
    for _ in range(60):
        mid = (lo + hi) / 2
        if stock_k(p, mid).sum() / D_all.sum() < target: lo = mid
        else: hi = mid
    return hi

d_t = np.array([2., 0., 3.])
print("test: mixed ->", {k: round(float(v), 3) for k, v in summarize(np.array([1., 1., 3.]), d_t).items()})  # fill 0.8, stockout 20, short days 50, overstock 20, ratio 1

EMP90 = emp_q(Y_train, 0.9)
FLAT = {"moving average (mean)": P_pt["moving average (28d)"], "moving median": P_pt["moving median (28d)"],
        "LightGBM P50": Q["q50"], "LightGBM P90": Q["q90"], "empirical P90 (28d)": EMP90}
FLAT = {k: v.ravel().astype("float64") for k, v in FLAT.items()}
print("fractional stock/demand:", {k: round(float(v.sum() / D_all.sum()), 3) for k, v in FLAT.items()})
# expect P50 0.689, P90 2.153, empirical P90 2.006

pol = pd.DataFrame({k: summarize(stock_k(v, 1.0), D_all) for k, v in FLAT.items()}).T
print("\n=== policies, rounded stock ===")
display(pol.round(3))

TARGET = pol.loc["LightGBM P90", "stock / demand"]
eq = {}
for k_name in ("moving average (mean)", "moving median", "LightGBM P50", "empirical P90 (28d)"):
    k = scale_to(FLAT[k_name], TARGET)
    eq[k_name] = {"k": k, **summarize(stock_k(FLAT[k_name], k), D_all)}
eq["LightGBM P90"] = {"k": 1.0, **summarize(stock_k(FLAT["LightGBM P90"], 1.0), D_all)}
print("\n=== equal total stock (target stock/demand %.3f) ===" % TARGET)
display(pd.DataFrame(eq).T.round(3))

test: mixed -> {'fill rate': 0.8, 'stockout units (% of demand)': 20.0, 'short on sales days (%)': 50.0, 'overstock units (% of demand)': 20.0, 'stock / demand': 1.0}
fractional stock/demand: {'moving average (mean)': 0.955, 'moving median': 0.716, 'LightGBM P50': 0.689, 'LightGBM P90': 2.153, 'empirical P90 (28d)': 2.006}

=== policies, rounded stock ===


,fill rate,stockout units (% of demand),short on sales days (%),overstock units (% of demand),stock / demand
moving average (mean),0.609,39.124,52.853,33.612,0.945
moving median,0.521,47.893,64.439,22.681,0.748
LightGBM P50,0.506,49.427,69.124,17.782,0.684
LightGBM P90,0.892,10.841,14.663,125.971,2.151
empirical P90 (28d),0.846,15.440,21.676,111.498,1.961



=== equal total stock (target stock/demand 2.151) ===


,k,fill rate,stockout units (% of demand),short on sales days (%),overstock units (% of demand),stock / demand
moving average (mean),2.273,0.845,15.510,23.033,130.662,2.152
moving median,3.000,0.750,24.981,38.934,142.932,2.180
LightGBM P50,3.133,0.776,22.365,38.083,137.496,2.151
empirical P90 (28d),1.100,0.866,13.434,19.367,130.048,2.166
LightGBM P90,1.000,0.892,10.841,14.663,125.971,2.151


In [8]:
# Cell 7: Save the final tables

REPORTS = Path("../reports")
REPORTS.mkdir(exist_ok=True)

qrows = []
for a, name in ((0.1, "q10"), (0.5, "q50"), (0.9, "q90")):
    for label, Pq in (("LightGBM (clipped)", Q[name]), ("empirical 28d", emp_q(Y_train, a))):
        p = Pq.ravel().astype("float64")
        qrows.append({"quantile": a, "model": label, "pinball": pinball(y, p, a),
                      "P(y < pred)": float(np.mean(y < p)), "P(y <= pred)": float(np.mean(y <= p)),
                      "total pred / total actual": float(p.sum() / y.sum())})
quant_tbl = pd.DataFrame(qrows)

boot_tbl = pd.DataFrame(rows)
assert "difference" in boot_tbl.columns and len(boot_tbl) == 4, "run Cell 4 again"

out = {"final_test_point_forecasts.csv": test_tbl, "final_test_levels.csv": lv,
       "final_test_quantiles.csv": quant_tbl, "final_test_bootstrap.csv": boot_tbl,
       "final_test_inventory.csv": pol, "final_test_inventory_equal_stock.csv": pd.DataFrame(eq).T}
for fname, tbl in out.items():
    tbl.round(4).to_csv(REPORTS / fname)
    print("saved", (REPORTS / fname).resolve().name, "| rows:", len(tbl))

saved final_test_point_forecasts.csv | rows: 7
saved final_test_levels.csv | rows: 3
saved final_test_quantiles.csv | rows: 6
saved final_test_bootstrap.csv | rows: 4
saved final_test_inventory.csv | rows: 5
saved final_test_inventory_equal_stock.csv | rows: 5


## Summary of Notebook 10: final test (d_1914-d_1941, 2016-04-25 to 2016-05-22)

**Protocol**
- Test window scored once; models and settings fixed before scoring (Poisson 300 rounds; P10/P50/P90 200 rounds; four horizon blocks, item_id, 63 leaves, learning rate 0.1); trained on rows up to d_1913
- Quantile predictions sorted per cell, then clipped at 0 (decided before scoring; 3,944 P10 cells and 16 P50 cells were negative)
- 134,032 units of demand; one window, so no fold-to-fold spread

**Point forecasts (test | validation mean of 3 folds)**

| Model | MAE | RMSE | WAPE | MASE | validation WAPE |
|---|---|---|---|---|---|
| seasonal naive (7d) | 1.363 | 2.737 | 0.868 | 1.112 | 0.877 |
| moving average (28d) | 1.174 | 2.347 | 0.748 | 0.976 | 0.748 |
| **LightGBM Poisson** | 1.128 | 2.202 | **0.718** | 0.961 | 0.739 |
| moving median (28d) | 1.106 | 2.452 | 0.704 | 0.852 | 0.698 |
| LightGBM P50 | 1.062 | 2.304 | 0.676 | 0.836 | n/a |

Fair pairs: Poisson vs moving average -4.0% WAPE (bootstrap over items: -0.036 to -0.024); P50 vs moving median -4.0% (-0.034 to -0.022). Intervals reflect item sampling only.

**Aggregate error (WAPE):** LightGBM Poisson 0.718 item-day, 0.068 category-day, 0.052 store-day; moving average 0.748 / 0.160 / 0.151; seasonal naive 0.868 / 0.087 / 0.072. Total forecast / actual: 0.949 (validation 0.997); store-day WAPE is almost entirely this bias.

**Quantiles:** pinball LightGBM vs empirical 28d: P10 0.152 vs 0.152, P50 0.531 vs 0.553, P90 0.355 vs 0.392 (-9.4%). P90 coverage 0.883-0.885 (validation 0.899).

**Inventory (rounded stock, 134,032 units demand)**
- Moving average stock: fill 0.609, stockout units 39.1% of demand, 0.945 x demand
- LightGBM P90 stock: fill 0.892, stockout units 10.8% (72% fewer), short on 14.7% of sales days, overstock 126% of demand, 2.151 x demand
- Equal total stock (2.151 x demand): P90 stockout units 10.8% vs 15.5% (scaled moving average, 30% fewer) vs 13.4% (scaled empirical P90, 19% fewer)

**Limits (for the README)**
- One test window of 4 weeks in spring (includes Mother's Day and the first days of May); results may differ in other seasons
- Bootstrap intervals assume independent items and ignore window-to-window variation
- Poisson under-forecast the total by 5.1% on this window; P90 coverage 88.5% instead of 90%
- ETS/ARIMA were evaluated only on a 300-item validation sample, not on the test window
- Observed sales stand in for demand; simulation has no costs, no carry-over
- Item level accuracy is hard: WAPE about 0.7 for every method; the clear gain is on aggregates